# Silver

Builds every Silver table from RAW, in dependency order. Run top to bottom as role **TRANSFORMER** on **TRANSFORMING_WH**. Every cell is SQL.

Each table has three cells: the quirks that apply, the `CREATE OR REPLACE TABLE` cell, and a check cell. A check counts the **violating rows**, the same way a dbt test works. The check cell lists every check with its count, and any check with failures raises `CHECK FAILED: <check> (<n> rows)`, which stops *Run all* before downstream tables build on a broken one. To see the bad rows, copy the check's condition into a `SELECT *`.

Silver keeps the full delivery history: Salesforce re-sends the whole record on every day it changes, and choosing the current state is a modeling decision for dbt. No business logic and no filtering here: deleted records stay, with `is_deleted`.

## Naming
- Primary key `<object>_id`. Foreign keys use the name of the key they reference, prefixed with the role when the name alone is ambiguous (`owner_user_id`, `converted_contact_id`, `parent_campaign_id`).
- Timestamps end in `_at` (`TIMESTAMP_TZ`, from Salesforce ISO strings with offset), dates in `_on` (`DATE`).
- Booleans start with `is_` / `has_`.
- Money is `NUMBER(18, 2)` in the record's own currency.
- Picklists that would collide with SQL keywords or generic names get the object as a prefix (`campaign_type`, `lead_status`, `member_status`).
- Every Salesforce table keeps `delivered_on` (date in `_file`), `_file`, `_file_row`, `_loaded_at`. Order history by `delivered_on` then `_file_row`, never by `_loaded_at`.

## Checks
Every Salesforce check cell starts with the same six checks:
- `not_null(<id>, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)`: key, lineage and parsed timestamps are filled. A NULL here usually means a wrong JSON path or a timestamp format that didn't parse.
- `unique(<id>, delivered_on)`: the grain, one row per record per delivery. Assumes one file per object per day.
- `row count = distinct (Id, file) in RAW`: the `QUALIFY` only dropped repeats within a file; no record lost, none doubled.
- `delivered_on = date in _file`: same expression as the build, so in practice it catches file names without a date.
- `delivered on the day it changed`: a daily file only holds records modified that day (date of `system_modstamp_at` = `delivered_on`). A failure points at the extract, not at this SQL.
- `created_at <= system_modstamp_at`: a record can't change before it exists.

Checks specific to a table have a comment above them in the check cell. OPPORTUNITY_HISTORY and ECB.RATES have a different grain, so every check there is commented.

How the checks are written:
- `NOT (cond) OR (cond) IS NULL` counts rows where the condition is false or can't be evaluated. `NOT (cond)` alone is NULL when an input is NULL, and `COUNT_IF` skips NULLs, so those rows would pass.
- `x NOT IN (...) OR x IS NULL` does the same for picklists: an unknown value or a missing one fails.
- `<fk> exists in <TABLE>` skips NULL keys; whether a key may be NULL is a separate `not_null` check.
- The failure message is cast to `int`, which can't succeed, so Snowflake raises an error that carries the message. That is how a SQL cell fails on purpose.

In [ ]:
ALTER SESSION SET TIMEZONE = 'UTC';
CREATE SCHEMA IF NOT EXISTS SILVER.SALESFORCE;
CREATE SCHEMA IF NOT EXISTS SILVER.ECB;

## SILVER.SALESFORCE.USERS
Columns: `user_id`, `user_name`, `email`, `user_role_id`, `is_active`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Sales reps. Re-delivered when a rep is deactivated.
- One row per user per delivery; a record repeated in one file keeps its last `_file_row` (applies to every object below).

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.USERS AS
SELECT
    RAW:Id::string AS user_id,
    RAW:Name::string AS user_name,
    RAW:Email::string AS email,
    RAW:UserRoleId::string AS user_role_id,
    RAW:IsActive::boolean AS is_active,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.USERS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(user_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(user_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.USERS
    UNION ALL SELECT 'unique(user_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.USERS GROUP BY user_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.USERS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.USERS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.USERS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.USERS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.USERS
    -- Descriptive fields every rep has.
    UNION ALL SELECT 'not_null(user_name, is_active)', COUNT_IF(user_name IS NULL OR is_active IS NULL) FROM SILVER.SALESFORCE.USERS
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.ACCOUNTS
Columns: `account_id`, `account_name`, `billing_country`, `industry`, `number_of_employees`, `owner_user_id`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Created when a lead converts.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.ACCOUNTS AS
SELECT
    RAW:Id::string AS account_id,
    RAW:Name::string AS account_name,
    RAW:BillingCountry::string AS billing_country,
    RAW:Industry::string AS industry,
    RAW:NumberOfEmployees::number AS number_of_employees,
    RAW:OwnerId::string AS owner_user_id,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.ACCOUNTS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(account_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(account_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    UNION ALL SELECT 'unique(account_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.ACCOUNTS GROUP BY account_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.ACCOUNTS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.ACCOUNTS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    -- Every account has a name and an owning rep.
    UNION ALL SELECT 'not_null(account_name, owner_user_id)', COUNT_IF(account_name IS NULL OR owner_user_id IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    -- Company size is always filled and positive; NULL fails too.
    UNION ALL SELECT 'number_of_employees > 0', COUNT_IF(NOT (number_of_employees > 0) OR (number_of_employees > 0) IS NULL) FROM SILVER.SALESFORCE.ACCOUNTS
    -- The owner is a known rep, in any delivery of USERS.
    UNION ALL SELECT 'owner_user_id exists in USERS', COUNT(*) FROM SILVER.SALESFORCE.ACCOUNTS c WHERE c.owner_user_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.USERS p WHERE p.user_id = c.owner_user_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.CAMPAIGNS
Columns: `campaign_id`, `campaign_name`, `campaign_type`, `campaign_status`, `start_on`, `end_on`, `budgeted_cost`, `actual_cost`, `parent_campaign_id`, `is_active`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Re-delivered as the status moves Planned → In Progress → Completed and when the actual cost is booked.
- Missing costs (quirk 6): some campaigns have no `actual_cost`, a few have neither cost. Kept as NULL; the fallback is a dbt decision.
- Costs are in EUR.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.CAMPAIGNS AS
SELECT
    RAW:Id::string AS campaign_id,
    RAW:Name::string AS campaign_name,
    RAW:Type::string AS campaign_type,
    RAW:Status::string AS campaign_status,
    RAW:StartDate::date AS start_on,
    RAW:EndDate::date AS end_on,
    RAW:BudgetedCost::number(18, 2) AS budgeted_cost,
    RAW:ActualCost::number(18, 2) AS actual_cost,
    RAW:ParentId::string AS parent_campaign_id,
    RAW:IsActive::boolean AS is_active,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.CAMPAIGNS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(campaign_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(campaign_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    UNION ALL SELECT 'unique(campaign_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.CAMPAIGNS GROUP BY campaign_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGNS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.CAMPAIGNS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- Every campaign has a name and a date range. Costs may be NULL (quirk 6).
    UNION ALL SELECT 'not_null(campaign_name, start_on, end_on)', COUNT_IF(campaign_name IS NULL OR start_on IS NULL OR end_on IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- Known picklist values only, so a new type from Salesforce stops the run instead of slipping through.
    UNION ALL SELECT 'campaign_type in (''Email'', ''Paid Search'', ''Webinar'', ''Partner'', ''Content'', ''Event'')', COUNT_IF(campaign_type NOT IN ('Email', 'Paid Search', 'Webinar', 'Partner', 'Content', 'Event') OR campaign_type IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- Known statuses only.
    UNION ALL SELECT 'campaign_status in (''Planned'', ''In Progress'', ''Completed'')', COUNT_IF(campaign_status NOT IN ('Planned', 'In Progress', 'Completed') OR campaign_status IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- A campaign can't end before it starts.
    UNION ALL SELECT 'start_on <= end_on', COUNT_IF(NOT (start_on <= end_on) OR (start_on <= end_on) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- NULL costs are allowed (quirk 6): COALESCE to 0, so only negative costs fail.
    UNION ALL SELECT 'costs >= 0', COUNT_IF(NOT (COALESCE(budgeted_cost, 0) >= 0 AND COALESCE(actual_cost, 0) >= 0) OR (COALESCE(budgeted_cost, 0) >= 0 AND COALESCE(actual_cost, 0) >= 0) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGNS
    -- Self-reference: a parent campaign is itself a campaign.
    UNION ALL SELECT 'parent_campaign_id exists in CAMPAIGNS', COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGNS c WHERE c.parent_campaign_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CAMPAIGNS p WHERE p.campaign_id = c.parent_campaign_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.LEADS
Columns: `lead_id`, `first_name`, `last_name`, `email`, `company`, `country`, `lead_source`, `lead_status`, `lead_status_raw`, `is_converted`, `converted_contact_id`, `converted_account_id`, `converted_opportunity_id`, `converted_on`, `owner_user_id`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Re-delivered on every status change (quirk 1).
- Picklist change (quirk 8): `Working` was renamed `Contacted` partway through history. `lead_status` maps it; `lead_status_raw` keeps what Salesforce sent.
- Duplicate leads (quirk 2): about 2% share an email with another lead under a different `Id`. Kept; deduplicating people is a dbt decision.
- Deleted leads (quirk 9) arrive with `IsDeleted = true`. Kept.
- A converted lead stops changing (quirk 3); its history continues on the contact and opportunity.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.LEADS AS
SELECT
    RAW:Id::string AS lead_id,
    RAW:FirstName::string AS first_name,
    RAW:LastName::string AS last_name,
    RAW:Email::string AS email,
    RAW:Company::string AS company,
    RAW:Country::string AS country,
    RAW:LeadSource::string AS lead_source,
    IFF(RAW:Status::string = 'Working', 'Contacted', RAW:Status::string) AS lead_status,
    RAW:Status::string AS lead_status_raw,
    RAW:IsConverted::boolean AS is_converted,
    RAW:ConvertedContactId::string AS converted_contact_id,
    RAW:ConvertedAccountId::string AS converted_account_id,
    RAW:ConvertedOpportunityId::string AS converted_opportunity_id,
    RAW:ConvertedDate::date AS converted_on,
    RAW:OwnerId::string AS owner_user_id,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.LEADS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(lead_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(lead_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.LEADS
    UNION ALL SELECT 'unique(lead_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.LEADS GROUP BY lead_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.LEADS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.LEADS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.LEADS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.LEADS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- Fields every lead has.
    UNION ALL SELECT 'not_null(email, lead_source, lead_status, is_converted, owner_user_id)', COUNT_IF(email IS NULL OR lead_source IS NULL OR lead_status IS NULL OR is_converted IS NULL OR owner_user_id IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- After mapping `Working` to `Contacted` (quirk 8), only current statuses remain.
    UNION ALL SELECT 'lead_status in (''New'', ''Contacted'', ''MQL'', ''SQL'', ''Unqualified'', ''Converted'')', COUNT_IF(lead_status NOT IN ('New', 'Contacted', 'MQL', 'SQL', 'Unqualified', 'Converted') OR lead_status IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- What Salesforce sent, so `Working` is still valid here.
    UNION ALL SELECT 'lead_status_raw in (''New'', ''Contacted'', ''MQL'', ''SQL'', ''Unqualified'', ''Converted'', ''Working'')', COUNT_IF(lead_status_raw NOT IN ('New', 'Contacted', 'MQL', 'SQL', 'Unqualified', 'Converted', 'Working') OR lead_status_raw IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- Known sources only.
    UNION ALL SELECT 'lead_source in (''Paid Search'', ''Webinar'', ''Partner'', ''Content'', ''Event'')', COUNT_IF(lead_source NOT IN ('Paid Search', 'Webinar', 'Partner', 'Content', 'Event') OR lead_source IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- `is_converted` is true exactly when the status is Converted.
    UNION ALL SELECT 'converted iff status Converted', COUNT_IF(NOT (is_converted = (lead_status = 'Converted')) OR (is_converted = (lead_status = 'Converted')) IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- Converting a lead creates a contact and an account and stamps the date.
    UNION ALL SELECT 'converted leads have contact, account and date', COUNT_IF(NOT (NOT is_converted OR (converted_contact_id IS NOT NULL AND converted_account_id IS NOT NULL AND converted_on IS NOT NULL)) OR (NOT is_converted OR (converted_contact_id IS NOT NULL AND converted_account_id IS NOT NULL AND converted_on IS NOT NULL)) IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- A lead can't be delivered as converted before its conversion date.
    UNION ALL SELECT 'converted_on <= delivered_on', COUNT_IF(NOT (converted_on IS NULL OR converted_on <= delivered_on) OR (converted_on IS NULL OR converted_on <= delivered_on) IS NULL) FROM SILVER.SALESFORCE.LEADS
    -- The owner is a known rep.
    UNION ALL SELECT 'owner_user_id exists in USERS', COUNT(*) FROM SILVER.SALESFORCE.LEADS c WHERE c.owner_user_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.USERS p WHERE p.user_id = c.owner_user_id)
    -- The account created on conversion was delivered. `converted_contact_id` is checked in the CONTACTS cell, which builds later.
    UNION ALL SELECT 'converted_account_id exists in ACCOUNTS', COUNT(*) FROM SILVER.SALESFORCE.LEADS c WHERE c.converted_account_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.ACCOUNTS p WHERE p.account_id = c.converted_account_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.CONTACTS
Columns: `contact_id`, `account_id`, `first_name`, `last_name`, `email`, `lead_source`, `mailing_country`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Created when a lead converts; the lead's `converted_contact_id` points here.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.CONTACTS AS
SELECT
    RAW:Id::string AS contact_id,
    RAW:AccountId::string AS account_id,
    RAW:FirstName::string AS first_name,
    RAW:LastName::string AS last_name,
    RAW:Email::string AS email,
    RAW:LeadSource::string AS lead_source,
    RAW:MailingCountry::string AS mailing_country,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.CONTACTS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(contact_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(contact_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.CONTACTS
    UNION ALL SELECT 'unique(contact_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.CONTACTS GROUP BY contact_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.CONTACTS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.CONTACTS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.CONTACTS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.CONTACTS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.CONTACTS
    -- Every contact belongs to an account and has an email.
    UNION ALL SELECT 'not_null(account_id, email)', COUNT_IF(account_id IS NULL OR email IS NULL) FROM SILVER.SALESFORCE.CONTACTS
    -- The contact's account was delivered.
    UNION ALL SELECT 'account_id exists in ACCOUNTS', COUNT(*) FROM SILVER.SALESFORCE.CONTACTS c WHERE c.account_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.ACCOUNTS p WHERE p.account_id = c.account_id)
    -- Foreign key from LEADS, checked here because CONTACTS builds after LEADS.
    UNION ALL SELECT 'every converted lead contact exists', COUNT(*) FROM SILVER.SALESFORCE.LEADS l WHERE l.converted_contact_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CONTACTS c WHERE c.contact_id = l.converted_contact_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.CAMPAIGN_MEMBERS
Columns: `campaign_member_id`, `campaign_id`, `lead_id`, `contact_id`, `member_status`, `has_responded`, `first_responded_on`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- A lead can be a member of several campaigns (quirk 4): one row per membership, not per lead.
- `contact_id` gets filled in when the lead converts, so the same member is delivered with NULL and later with a contact.
- `first_responded_on` is a `DATE`, `created_at` a timestamp; don't cast one to the other's type.
- Missing response dates (quirk 10): about 1% of responded members have no `first_responded_on`. Kept as NULL.
- Touches after the deal (quirk 11) are kept; excluding them is attribution logic.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.CAMPAIGN_MEMBERS AS
SELECT
    RAW:Id::string AS campaign_member_id,
    RAW:CampaignId::string AS campaign_id,
    RAW:LeadId::string AS lead_id,
    RAW:ContactId::string AS contact_id,
    RAW:Status::string AS member_status,
    RAW:HasResponded::boolean AS has_responded,
    RAW:FirstRespondedDate::date AS first_responded_on,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.CAMPAIGN_MEMBERS
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(campaign_member_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(campaign_member_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    UNION ALL SELECT 'unique(campaign_member_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS GROUP BY campaign_member_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.CAMPAIGN_MEMBERS
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- Every membership has a campaign and a status.
    UNION ALL SELECT 'not_null(campaign_id, member_status, has_responded)', COUNT_IF(campaign_id IS NULL OR member_status IS NULL OR has_responded IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- A member is a person: a lead before conversion, a contact after, or both.
    UNION ALL SELECT 'has a lead or a contact', COUNT_IF(NOT (lead_id IS NOT NULL OR contact_id IS NOT NULL) OR (lead_id IS NOT NULL OR contact_id IS NOT NULL) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- Known statuses only.
    UNION ALL SELECT 'member_status in (''Sent'', ''Responded'')', COUNT_IF(member_status NOT IN ('Sent', 'Responded') OR member_status IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- `has_responded` is true exactly when the status is Responded.
    UNION ALL SELECT 'responded iff status Responded', COUNT_IF(NOT (has_responded = (member_status = 'Responded')) OR (has_responded = (member_status = 'Responded')) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- The date may be missing for a responder (quirk 10), but never set for a non-responder.
    UNION ALL SELECT 'first_responded_on only when responded', COUNT_IF(NOT (first_responded_on IS NULL OR has_responded) OR (first_responded_on IS NULL OR has_responded) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- A response can't be delivered before it happened.
    UNION ALL SELECT 'first_responded_on <= delivered_on', COUNT_IF(NOT (first_responded_on IS NULL OR first_responded_on <= delivered_on) OR (first_responded_on IS NULL OR first_responded_on <= delivered_on) IS NULL) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS
    -- Once a conversion fills `contact_id`, later deliveries of the same member keep it. LAG compares each delivery with the one before.
    UNION ALL SELECT 'contact_id never reverts to NULL', COUNT(*) FROM (SELECT contact_id, LAG(contact_id) OVER (PARTITION BY campaign_member_id ORDER BY delivered_on, _file_row) AS prev FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS) WHERE prev IS NOT NULL AND contact_id IS NULL
    -- Foreign keys: campaign, lead and contact were all delivered.
    UNION ALL SELECT 'campaign_id exists in CAMPAIGNS', COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS c WHERE c.campaign_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CAMPAIGNS p WHERE p.campaign_id = c.campaign_id)
    UNION ALL SELECT 'lead_id exists in LEADS', COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS c WHERE c.lead_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.LEADS p WHERE p.lead_id = c.lead_id)
    UNION ALL SELECT 'contact_id exists in CONTACTS', COUNT(*) FROM SILVER.SALESFORCE.CAMPAIGN_MEMBERS c WHERE c.contact_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CONTACTS p WHERE p.contact_id = c.contact_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.OPPORTUNITIES
Columns: `opportunity_id`, `account_id`, `campaign_id`, `opportunity_name`, `opportunity_type`, `stage_name`, `probability`, `amount`, `currency`, `close_on`, `is_closed`, `is_won`, `owner_user_id`, `lead_source`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Re-delivered on every stage change. Stage regressions (quirk 5) show up as a later delivery with an earlier stage.
- `amount` is in `currency` (quirk 7): EUR, GBP or USD. Conversion with `SILVER.ECB.RATES` happens in dbt.
- `campaign_id` is Salesforce's primary campaign source, the fallback when there's no contact role (quirk 12). Can be NULL.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.OPPORTUNITIES AS
SELECT
    RAW:Id::string AS opportunity_id,
    RAW:AccountId::string AS account_id,
    RAW:CampaignId::string AS campaign_id,
    RAW:Name::string AS opportunity_name,
    RAW:Type::string AS opportunity_type,
    RAW:StageName::string AS stage_name,
    RAW:Probability::number AS probability,
    RAW:Amount::number(18, 2) AS amount,
    UPPER(RAW:CurrencyIsoCode::string) AS currency,
    RAW:CloseDate::date AS close_on,
    RAW:IsClosed::boolean AS is_closed,
    RAW:IsWon::boolean AS is_won,
    RAW:OwnerId::string AS owner_user_id,
    RAW:LeadSource::string AS lead_source,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.OPPORTUNITIES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(opportunity_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(opportunity_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    UNION ALL SELECT 'unique(opportunity_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITIES GROUP BY opportunity_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITIES) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.OPPORTUNITIES
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Fields every opportunity has. `campaign_id` may be NULL (quirk 12).
    UNION ALL SELECT 'not_null(account_id, stage_name, amount, currency, close_on, is_closed, is_won, owner_user_id)', COUNT_IF(account_id IS NULL OR stage_name IS NULL OR amount IS NULL OR currency IS NULL OR close_on IS NULL OR is_closed IS NULL OR is_won IS NULL OR owner_user_id IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Known stages only.
    UNION ALL SELECT 'stage_name in (''Discovery'', ''Demo'', ''Proposal'', ''Negotiation'', ''Closed Won'', ''Closed Lost'')', COUNT_IF(stage_name NOT IN ('Discovery', 'Demo', 'Proposal', 'Negotiation', 'Closed Won', 'Closed Lost') OR stage_name IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- After UPPER, only the three currencies ECB rates are loaded for (quirk 7).
    UNION ALL SELECT 'currency in (''EUR'', ''GBP'', ''USD'')', COUNT_IF(currency NOT IN ('EUR', 'GBP', 'USD') OR currency IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Known types only.
    UNION ALL SELECT 'opportunity_type in (''New Business'', ''Existing Business'')', COUNT_IF(opportunity_type NOT IN ('New Business', 'Existing Business') OR opportunity_type IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Every deal has a positive amount; NULL fails too.
    UNION ALL SELECT 'amount > 0', COUNT_IF(NOT (amount > 0) OR (amount > 0) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Probability is a percentage.
    UNION ALL SELECT 'probability between 0 and 100', COUNT_IF(NOT (probability BETWEEN 0 AND 100) OR (probability BETWEEN 0 AND 100) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- `is_closed` is true exactly when the stage is Closed Won or Closed Lost.
    UNION ALL SELECT 'is_closed iff stage Closed *', COUNT_IF(NOT (is_closed = (stage_name IN ('Closed Won', 'Closed Lost'))) OR (is_closed = (stage_name IN ('Closed Won', 'Closed Lost'))) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- `is_won` is true exactly when the stage is Closed Won.
    UNION ALL SELECT 'is_won iff stage Closed Won', COUNT_IF(NOT (is_won = (stage_name = 'Closed Won')) OR (is_won = (stage_name = 'Closed Won')) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITIES
    -- Foreign keys: account, primary campaign and owner were all delivered.
    UNION ALL SELECT 'account_id exists in ACCOUNTS', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITIES c WHERE c.account_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.ACCOUNTS p WHERE p.account_id = c.account_id)
    UNION ALL SELECT 'campaign_id exists in CAMPAIGNS', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITIES c WHERE c.campaign_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CAMPAIGNS p WHERE p.campaign_id = c.campaign_id)
    UNION ALL SELECT 'owner_user_id exists in USERS', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITIES c WHERE c.owner_user_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.USERS p WHERE p.user_id = c.owner_user_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
Columns: `opportunity_contact_role_id`, `opportunity_id`, `contact_id`, `contact_role`, `is_primary`, `is_deleted`, `created_at`, `last_modified_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Links the people on a deal to the opportunity; the bridge from campaign touches to pipeline.
- A few opportunities have no contact role (quirk 12). That's a gap in this table, not a check failure.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES AS
SELECT
    RAW:Id::string AS opportunity_contact_role_id,
    RAW:OpportunityId::string AS opportunity_id,
    RAW:ContactId::string AS contact_id,
    RAW:Role::string AS contact_role,
    RAW:IsPrimary::boolean AS is_primary,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:LastModifiedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS last_modified_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id, _file ORDER BY _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    SELECT 'not_null(opportunity_contact_role_id, delivered_on, created_at, is_deleted, _file, _file_row, _loaded_at)', COUNT_IF(opportunity_contact_role_id IS NULL OR delivered_on IS NULL OR created_at IS NULL OR is_deleted IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    UNION ALL SELECT 'unique(opportunity_contact_role_id, delivered_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES GROUP BY opportunity_contact_role_id, delivered_on HAVING COUNT(*) > 1)
    UNION ALL SELECT 'row count = distinct (Id, file) in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES) - COUNT(DISTINCT RAW:Id, _file)) FROM RAW.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    UNION ALL SELECT 'delivered on the day it changed', COUNT_IF(NOT (system_modstamp_at::date = delivered_on) OR (system_modstamp_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    UNION ALL SELECT 'created_at <= system_modstamp_at', COUNT_IF(NOT (created_at <= system_modstamp_at) OR (created_at <= system_modstamp_at) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    -- A role always links one opportunity to one contact.
    UNION ALL SELECT 'not_null(opportunity_id, contact_id, is_primary)', COUNT_IF(opportunity_id IS NULL OR contact_id IS NULL OR is_primary IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    -- Known roles only.
    UNION ALL SELECT 'contact_role in (''Decision Maker'', ''Influencer'')', COUNT_IF(contact_role NOT IN ('Decision Maker', 'Influencer') OR contact_role IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES
    -- Foreign keys: the opportunity and the contact were both delivered.
    UNION ALL SELECT 'opportunity_id exists in OPPORTUNITIES', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES c WHERE c.opportunity_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITIES p WHERE p.opportunity_id = c.opportunity_id)
    UNION ALL SELECT 'contact_id exists in CONTACTS', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITY_CONTACT_ROLES c WHERE c.contact_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.CONTACTS p WHERE p.contact_id = c.contact_id)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.SALESFORCE.OPPORTUNITY_HISTORY
Columns: `opportunity_history_id`, `opportunity_id`, `stage_name`, `amount`, `close_on`, `probability`, `is_deleted`, `created_at`, `system_modstamp_at`, `delivered_on`, `_file`, `_file_row`, `_loaded_at`

- Append-only: one row per stage change, never re-delivered as new state. The grain is the history `Id`, not `Id` + delivery. If a row ever arrives twice, keep its first delivery.
- Has no `LastModifiedDate`.
- `amount` is in the opportunity's currency.

In [ ]:
CREATE OR REPLACE TABLE SILVER.SALESFORCE.OPPORTUNITY_HISTORY AS
SELECT
    RAW:Id::string AS opportunity_history_id,
    RAW:OpportunityId::string AS opportunity_id,
    RAW:StageName::string AS stage_name,
    RAW:Amount::number(18, 2) AS amount,
    RAW:CloseDate::date AS close_on,
    RAW:Probability::number AS probability,
    RAW:IsDeleted::boolean AS is_deleted,
    TO_TIMESTAMP_TZ(RAW:CreatedDate::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS created_at,
    TO_TIMESTAMP_TZ(RAW:SystemModstamp::string, 'YYYY-MM-DD"T"HH24:MI:SS.FF3TZHTZM') AS system_modstamp_at,
    TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}')) AS delivered_on,
    _file,
    _file_row,
    _loaded_at
FROM RAW.SALESFORCE.OPPORTUNITY_HISTORY
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:Id ORDER BY delivered_on, _file_row) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    -- Key, parent, stage, amount and lineage are filled.
    SELECT 'not_null(opportunity_history_id, opportunity_id, stage_name, amount, created_at, delivered_on, _file, _file_row, _loaded_at)', COUNT_IF(opportunity_history_id IS NULL OR opportunity_id IS NULL OR stage_name IS NULL OR amount IS NULL OR created_at IS NULL OR delivered_on IS NULL OR _file IS NULL OR _file_row IS NULL OR _loaded_at IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY
    -- The grain is the history Id alone: append-only, one row per stage change.
    UNION ALL SELECT 'unique(opportunity_history_id)', COUNT(*) FROM (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY GROUP BY opportunity_history_id HAVING COUNT(*) > 1)
    -- The `QUALIFY` kept exactly one row per Id: the first delivery.
    UNION ALL SELECT 'row count = distinct Id in RAW', ABS((SELECT COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY) - COUNT(DISTINCT RAW:Id)) FROM RAW.SALESFORCE.OPPORTUNITY_HISTORY
    -- Same expression as the build; in practice it catches file names without a date.
    UNION ALL SELECT 'delivered_on = date in _file', COUNT_IF(NOT (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) OR (delivered_on = TO_DATE(REGEXP_SUBSTR(_file, '[0-9]{4}-[0-9]{2}-[0-9]{2}'))) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY
    -- A history row is delivered once, on the day it was written. No LastModifiedDate, so created_at is the date to compare.
    UNION ALL SELECT 'delivered on the day it was created', COUNT_IF(NOT (created_at::date = delivered_on) OR (created_at::date = delivered_on) IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY
    -- Same stages as OPPORTUNITIES.
    UNION ALL SELECT 'stage_name in (''Discovery'', ''Demo'', ''Proposal'', ''Negotiation'', ''Closed Won'', ''Closed Lost'')', COUNT_IF(stage_name NOT IN ('Discovery', 'Demo', 'Proposal', 'Negotiation', 'Closed Won', 'Closed Lost') OR stage_name IS NULL) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY
    -- Every history row belongs to a delivered opportunity.
    UNION ALL SELECT 'opportunity_id exists in OPPORTUNITIES', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY c WHERE c.opportunity_id IS NOT NULL AND NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITIES p WHERE p.opportunity_id = c.opportunity_id)
    -- Every delivery of an opportunity has a history row from the same day with the same stage, so the two tables tell the same story.
    UNION ALL SELECT 'one history row per opportunity delivery', COUNT(*) FROM SILVER.SALESFORCE.OPPORTUNITIES o WHERE NOT EXISTS (SELECT 1 FROM SILVER.SALESFORCE.OPPORTUNITY_HISTORY h WHERE h.opportunity_id = o.opportunity_id AND h.delivered_on = o.delivered_on AND h.stage_name = o.stage_name)
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## SILVER.ECB.RATES
Columns: `currency`, `rate_on`, `rate`

- Shared with saas-revenue-analytics; same definition as its notebook, so either project can rebuild it.
- Units of currency per 1 EUR. GBP and USD only; EUR is the base.
- Daily extracts overlap: one row per currency and date, latest file wins.
- `OBS_VALUE` is a string in RAW.

In [ ]:
CREATE OR REPLACE TABLE SILVER.ECB.RATES AS
SELECT
    RAW:CURRENCY::string AS currency,
    RAW:TIME_PERIOD::date AS rate_on,
    RAW:OBS_VALUE::number(18, 6) AS rate
FROM RAW.ECB.RATES
QUALIFY ROW_NUMBER() OVER (PARTITION BY RAW:CURRENCY, RAW:TIME_PERIOD ORDER BY _file DESC, _file_row DESC) = 1;

In [ ]:
WITH checks (check_name, failures) AS (
    -- Every rate has a currency, a date and a value.
    SELECT 'not_null(currency, rate_on, rate)', COUNT_IF(currency IS NULL OR rate_on IS NULL OR rate IS NULL) FROM SILVER.ECB.RATES
    -- The grain: one rate per currency per day.
    UNION ALL SELECT 'unique(currency, rate_on)', COUNT(*) FROM (SELECT 1 FROM SILVER.ECB.RATES GROUP BY currency, rate_on HAVING COUNT(*) > 1)
    -- The `QUALIFY` kept exactly one row per currency and date from the overlapping extracts.
    UNION ALL SELECT 'row count = distinct (currency, date) in RAW', ABS((SELECT COUNT(*) FROM SILVER.ECB.RATES) - COUNT(DISTINCT RAW:CURRENCY, RAW:TIME_PERIOD)) FROM RAW.ECB.RATES
    -- Only the currencies opportunities use; EUR is the base and has no rate.
    UNION ALL SELECT 'currency in (''GBP'', ''USD'')', COUNT_IF(currency NOT IN ('GBP', 'USD') OR currency IS NULL) FROM SILVER.ECB.RATES
    -- A rate is always positive; NULL fails too (OBS_VALUE didn't parse).
    UNION ALL SELECT 'rate > 0', COUNT_IF(NOT (rate > 0) OR (rate > 0) IS NULL) FROM SILVER.ECB.RATES
    -- The ECB publishes no rates on weekends; a weekend date means a bad parse.
    UNION ALL SELECT 'business days only', COUNT_IF(NOT (DAYOFWEEKISO(rate_on) <= 5) OR (DAYOFWEEKISO(rate_on) <= 5) IS NULL) FROM SILVER.ECB.RATES
    -- Rates start on or before the earliest close date. It checks the start of the range only, not the end or gaps.
    UNION ALL SELECT 'covers every opportunity close date', COUNT(*) FROM (SELECT MIN(close_on) lo FROM SILVER.SALESFORCE.OPPORTUNITIES) o, (SELECT MIN(rate_on) lo FROM SILVER.ECB.RATES) r WHERE r.lo > o.lo
)
SELECT check_name, failures,
    IFF(failures = 0, 'pass', ('CHECK FAILED: ' || check_name || ' (' || failures || ' rows)')::int::string) AS result
FROM checks;

## Summary

In [ ]:
SELECT table_schema, table_name, row_count
FROM SILVER.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('SALESFORCE', 'ECB')
ORDER BY 1, 2;